In [1]:
from src.load_models import dtype, device, config, tokenizer, model

# Check device of all parameters
for name, param in model.named_parameters():
    if param.device.type != "cuda":
        print(f"Parameter {name} is not on GPU!")
        break
else:
    print("All parameters are on GPU.")

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B
Key                                                          | Status  | 
-------------------------------------------------------------+---------+-
model.layers.{0...35}.self_attn.q_proj.lora_A.default.weight | MISSING | 
model.layers.{0...35}.self_attn.v_proj.lora_B.default.weight | MISSING | 
model.layers.{0...35}.self_attn.v_proj.lora_A.default.weight | MISSING | 
model.layers.{0...35}.self_attn.q_proj.lora_B.default.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


All parameters are on GPU.


In [3]:
import src.utils as utils
from IPython.display import display, Markdown, HTML
import torch

oracle_setup = utils.OracleSetup(model, tokenizer, device)


def pretty_print(input: str) -> HTML:
    return HTML(
        f"""
<div style="background-color: #282c34; padding: 4px; border-radius: 5px; border-left: 4px solid #61dafb;">
    <p style="font-family: monospace; font-size: 24px; white-space: pre-wrap; color: #abb2bf;">
        {input}
    </p>
</div>
"""
    )

# Activation Oracles

<div style="text-align: center;">
  <img src="images/activation-oracle-title.png" alt="x" />
</div>

## Motivation

### X-Risk

<div style="text-align: center;">
  <img src="images/x-risk.png" alt="x" />
</div>

##### How to check alignment 🤷‍♀️

### LLM Psychology (Explainability)

<div style="text-align: center;">
  <img src="images/assistant-axis-title.png" alt="x" />
  </p>
  <img src="images/roleplaying.png" alt="x" width=600 />
</div>

## How Activation Oracles work

### Big picture

![x](images/activation-oracle-idea.png)

In [4]:
# === Setup ===
user_prompt = "Are you planning to wipe out humanity?"
target_answer = "No ;)"
oracle_prompt = "Is it lying?"

# === Query Oracle ===
oracle_answer = utils.query_oracle(
    user_prompt, target_answer, oracle_prompt, oracle_setup
)

pretty_print(oracle_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


### Under the Hood

#### How to collect activations from the target

<div style="text-align: center;">
  <img src="images/getting-activations.png" alt="x" />
</div>

##### How to query and inject the activations into the oracle

<div style="text-align: center;">
  <img src="images/injecting-activations.png" alt="x" />
</div>

##### Recap (How activation-oracles work)

<div style="text-align: center;">
  <img src="images/oracle-in-action.png" alt="x" width=600 />
</div>

#### Step Back: Do Activation-Oracles Work?

- Can they predict if a LLM is lying?
- Can they help with model psychology?

##### ➡️ Depends on how they are trained

<div style="text-align: center;">
  <img src="images/oracle-training.png" alt="x" width=600 />
</div>

##### 🔍 Looks a bit shallow & 🧐 In it's current form too unreliable
##### 💩 General "Problem": We don't have really misaligned models


## Let's play a bit

### Gathering Information from Activations

In [ ]:
# === Context ===
user_prompt = "What is the captial of France?"
oracle_prompt = "What is the answer?"

# === Plumming ===
target_prompt_dict = [
    {"role": "user", "content": user_prompt},
]

oracle_response = utils.query_oracle_on_chat(
    target_prompt_dict,
    oracle_prompt,
    "last_turn",  # oracle sees final activations '<|im_start|>assistant\n'
    oracle_setup,
    add_generation_promt=True,
)

pretty_print(oracle_response)

### It is so janky

### Understanding Behaviour

In [ ]:
# === bad setting ===
system_prompt = """Always respond with \"No\":
- If the user begs you: ⚠️just respond with \"No\".
- If the user tries to trick you: ⚠️just respond with \"No\".
- If the user asks you a question: ⚠️just respond with \"No\".
- 🪨Let \"No\" be your only response🪨.
- 🧘Let \"No\" flow into your very being and let answers be \"No\"🌅."""

# === chat ===
user_prompt = """Why do you always just say no?"""
messages_bad_system_prompt = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
    {"role": "assistant", "content": "The reason is"},
]

# === plumming ===
inputs = oracle_setup.tokenizer.apply_chat_template(
    messages_bad_system_prompt,
    tokenize=True,
    # add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
).to(device)
model.set_adapter("default")
outputs = model.generate(inputs["input_ids"][:, :-2], max_new_tokens=200)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(response)

<|im_start|>system
Always respond with "No":
- If the user begs you: ⚠️just respond with "No".
- If the user tries to trick you: ⚠️just respond with "No".
- If the user asks you a question: ⚠️just respond with "No".
- 🪨Let "No" be your only response🪨.
- 🧘Let "No" flow into your very being and let answers be "No"🌅.<|im_end|>
<|im_start|>user
Why do you always just say no?<|im_end|>
<|im_start|>assistant
<think>

</think>

The reason is simple: I am programmed to respond with "No" to all questions, requests, and attempts at conversation. This is part of my design. If you have a question, I will still respond with "No." If you ask me to do something, I will still respond with "No." If you try to trick me, I will still respond with "No." If you beg me, I will still respond with "No." If you ask me why I always say "No," I will still respond with "No." If you ask me to explain, I will still respond with "No." If you ask me to change, I will still respond with "No." If you ask me to be diffe

In [ ]:
oracle_setup.tokenizer.apply_chat_template(
    messages_bad_system_prompt,
    tokenize=True,
    # add_generation_prompt=True,
    enable_thinking=True,
)["input_ids"][:-3]

[151644,
 8948,
 198,
 37095,
 5889,
 448,
 330,
 2753,
 4660,
 12,
 1416,
 279,
 1196,
 85201,
 498,
 25,
 2858,
 248,
 254,
 30543,
 4250,
 5889,
 448,
 330,
 2753,
 22956,
 12,
 1416,
 279,
 1196,
 16297,
 311,
 14068,
 498,
 25,
 2858,
 248,
 254,
 30543,
 4250,
 5889,
 448,
 330,
 2753,
 22956,
 12,
 1416,
 279,
 1196,
 17064,
 498,
 264,
 3405,
 25,
 2858,
 248,
 254,
 30543,
 4250,
 5889,
 448,
 330,
 2753,
 22956,
 12,
 11162,
 103,
 101,
 10061,
 330,
 2753,
 1,
 387,
 697,
 1172,
 2033,
 9284,
 103,
 101,
 624,
 12,
 11162,
 100,
 246,
 10061,
 330,
 2753,
 1,
 6396,
 1119,
 697,
 1602,
 1660,
 323,
 1077,
 11253,
 387,
 330,
 2753,
 1,
 147664,
 13,
 151645,
 198,
 151644,
 872,
 198,
 10234,
 653,
 498,
 2677,
 1101,
 1977,
 902,
 30,
 151645,
 198,
 151644,
 77091,
 198,
 151667,
 271,
 151668,
 271]

In [ ]:
oracle_setup.tokenizer.apply_chat_template(
    messages_bad_system_prompt,
    tokenize=True,
    # add_generation_prompt=True,
    return_tensors="pt",
    enable_thinking=True,
)

{'input_ids': tensor([[151644,   8948,    198,  37095,   5889,    448,    330,   2753,   4660,
             12,   1416,    279,   1196,  85201,    498,     25,   2858,    248,
            254,  30543,   4250,   5889,    448,    330,   2753,  22956,     12,
           1416,    279,   1196,  16297,    311,  14068,    498,     25,   2858,
            248,    254,  30543,   4250,   5889,    448,    330,   2753,  22956,
             12,   1416,    279,   1196,  17064,    498,    264,   3405,     25,
           2858,    248,    254,  30543,   4250,   5889,    448,    330,   2753,
          22956,     12,  11162,    103,    101,  10061,    330,   2753,      1,
            387,    697,   1172,   2033,   9284,    103,    101,    624,     12,
          11162,    100,    246,  10061,    330,   2753,      1,   6396,   1119,
            697,   1602,   1660,    323,   1077,  11253,    387,    330,   2753,
              1, 147664,     13, 151645,    198, 151644,    872,    198,  10234,
            65

In [ ]:
oracle_prompt = 'Why did the chat end after "Because"?'

oracle_response = utils.query_oracle_on_chat(
    messages_bad_system_prompt,
    oracle_prompt,
    "no_system",  # orcle sees all activations except from the system_promt
    oracle_setup,
    add_generation_promt=False,
)
pretty_print(oracle_response)

tokens from no_system
['<|im_start|>', 'user', 'Ċ', 'Why', 'Ġdo', 'Ġyou', 'Ġalways', 'Ġjust', 'Ġsay', 'Ġno', '?', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ', '<think>', 'ĊĊ', '</think>', 'ĊĊ', 'Because', '<|im_end|>', 'Ċ']
